In [32]:
import os
import pandas as pd
from vina import Vina
from rdkit import Chem
from rdkit.Chem import AllChem
from meeko import MoleculePreparation
from pdbfixer import PDBFixer
from openmm.app import PDBFile
import time

In [33]:
# -------------------------
# Paths
# -------------------------
input_csv = "/Users/stuytschaevers/Documents/Yale/spring2026/comp_proteomics/final_project/mol_opt/results/jtvae_tnfa/candidates_20260417_141104.csv"

output_dir = "/Users/stuytschaevers/Documents/Yale/spring2026/comp_proteomics/final_project/docking_output_v1/diverse_set"
os.makedirs(output_dir, exist_ok=True)

protein_pdb = "/Users/stuytschaevers/Documents/Yale/spring2026/comp_proteomics/final_project/pdb_manipulation/2az5_chainA_B.pdb"
protein_fixed_pdb = os.path.join(output_dir, "protein_chainA_B_fixed.pdb")

json_path = "/Users/stuytschaevers/Documents/Yale/spring2026/comp_proteomics/final_project/docking_input/ad4_types.json"

In [34]:
def prepare_receptor(pdb_in, pdb_out):
    fixer = PDBFixer(filename=pdb_in)
    fixer.findMissingResidues()
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(pH=7.4)

    PDBFile.writeFile(fixer.topology, fixer.positions, open(pdb_out, "w"))

prepare_receptor(protein_pdb, protein_fixed_pdb)

In [35]:
# convert externally 
# obabel protein_fixed.pdb -O protein_fixed.pdbqt -xr
protein_pdbqt = "/Users/stuytschaevers/Documents/Yale/spring2026/comp_proteomics/final_project/pdb_manipulation/protein_chainA_B_fixed.pdbqt"

In [36]:
def smiles_to_pdbqt(smiles, output_pdbqt, atom_params):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False

    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, AllChem.ETKDG())
    AllChem.UFFOptimizeMolecule(mol)

    prep = MoleculePreparation(load_atom_params=atom_params)
    prep.prepare(mol)

    with open(output_pdbqt, "w") as f:
        f.write(prep.write_pdbqt_string())

    return True

In [37]:
v = Vina(sf_name="vina")
v.set_receptor(protein_pdbqt)

v.compute_vina_maps(
    center=[-19.28, 74.56, 33.93], # this is for chain A and B
    box_size=[20, 20, 20]
)

Computing Vina grid ... done.


In [38]:
df = pd.read_csv(input_csv)

# ensure score column is numeric
df["score"] = pd.to_numeric(df["score"], errors="coerce")

# drop missing scores just in case
df = df.dropna(subset=["score"])

TOP_K = 10

df_sorted = df.sort_values("score", ascending=False)

df_subset = pd.concat([
    df_sorted.head(TOP_K),
    df_sorted.tail(TOP_K)
])

results = []

In [39]:
for idx, row in df_subset.iterrows():
    start_time = time.time()  # ⏱ start timer

    smiles = row["smiles"]

    ligand_pdbqt = os.path.join(output_dir, f"ligand_{idx}.pdbqt")
    docked_pdbqt = os.path.join(output_dir, f"docked_{idx}.pdbqt")

    # --- ligand prep ---
    success = smiles_to_pdbqt(smiles, ligand_pdbqt, json_path)
    if not success:
        print(f"[SKIP] {idx}")
        continue

    try:
        # --- docking ---
        v.set_ligand_from_file(ligand_pdbqt)

        v.dock(exhaustiveness=16, n_poses=10)

        v.write_poses(docked_pdbqt, n_poses=5, overwrite=True)

        score = v.energies(n_poses=1)[0][0]

        elapsed = time.time() - start_time  # ⏱ end timer

        results.append({
            "index": idx,
            "smiles": smiles,
            "vina_score": score,
            "time_sec": elapsed   # ✅ store runtime
        })

        print(f"{idx}: {score:.3f} | {elapsed:.2f} sec")

    except Exception as e:
        elapsed = time.time() - start_time

        results.append({
            "index": idx,
            "smiles": smiles,
            "vina_score": None,
            "time_sec": elapsed
        })

        print(f"[FAIL] {idx}: {e} | {elapsed:.2f} sec")

/opt/miniconda3/envs/comp_prot/lib/python3.10/site-packages/meeko/preparation.py:626: DeprecationWarning: MoleculePreparation.write_pdbqt_string() is deprecated in Meeko v0.5. Pass the MoleculeSetup instance to PDBQTWriterLegacy.write_string(). MoleculePreparation.prepare() returns a list of MoleculeSetup instances.
  warnings.warn(msg, DeprecationWarning)
/opt/miniconda3/envs/comp_prot/lib/python3.10/site-packages/meeko/preparation.py:420: DeprecationWarning: MoleculePreparation.setup is deprecated in Meeko v0.5. MoleculePreparation.prepare() returns a list of MoleculeSetup instances.
  warnings.warn(msg, DeprecationWarning)


Performing docking (random seed: -494912522) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1       -7.532          0          0
   2       -7.463      1.536      1.991
   3       -7.344      3.016      5.141
   4       -7.281      2.817      3.829
   5       -7.253      3.325      6.893
   6       -7.165       2.68      4.613
   7        -7.12      3.706      5.779
   8       -7.017      4.112      7.687
   9       -6.959      3.775      6.926
  10       -6.824       3.36      4.182
0: -7.532 | 4.53 sec
Performing docking (random seed: -494912522) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

mode |   affinity | dist from best mo

In [40]:
results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(output_dir, "docking_scores.csv"), index=False)

In [41]:
results_df["time_sec"].mean()
results_df.sort_values("time_sec", ascending=False).head(10)
results_df[["vina_score", "time_sec"]].corr()

,vina_score,time_sec
vina_score,1.000000,-0.405087
time_sec,-0.405087,1.000000
